In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# استخدام حمل العمل

<span id="usage"></span>
الاستخدام هو مقياس للمدة التي يكون فيها QPU مقفلًا لحمل عملك، ويُحسب بطريقة مختلفة بحسب وضع التنفيذ الذي تستخدمه.

* استخدام الجلسة (Session) هو وقت الساعة الفعلي الذي تكون فيه الجلسة نشطة. راجع [طول الجلسة](/guides/run-jobs-session#session-length) للمزيد من المعلومات حول انتقالات حالة الجلسة.
* استخدام الدُفعة (Batch) هو مجموع الوقت الكمي (الوقت الذي يستغرقه مجمع QPU لمعالجة مهمتك) لجميع المهام في الدُفعة.
* استخدام المهمة الفردية هو الوقت الكمي الذي تستهلكه المهمة أثناء المعالجة.

لاحظ أن المهام الفاشلة أو الملغاة تُحسب ضمن استخدامك في ظروف معينة — راجع قسم [المهام الفاشلة والملغاة](#failed-job) للتفاصيل.

بالنسبة لمستخدمي الخطط المدفوعة، يحدد الاستخدام تكلفة حمل العمل. راجع [إدارة التكلفة](/guides/manage-cost) للتفاصيل.

<span id="failed-job"></span>
## الاستخدام للمهام الفاشلة والملغاة
عند فشل مهمة أو إلغائها، يكون الاستخدام المُبلَّغ عنه كالتالي:

* وضع المهمة أو الدُفعة: الاستخدام المُبلَّغ عنه هو المدة التي كان فيها QPU مقفلًا لتنفيذ حمل عملك حتى لحظة الفشل أو الإلغاء. لذلك، إذا حدث الفشل أو الإلغاء قبل القفل، يكون الاستخدام المُبلَّغ عنه صفرًا. وإلا، يكون الاستخدام المُبلَّغ عنه لحمل العمل هو مقدار الاستخدام قبل فشله أو إلغائه. وبهذا، لا تظهر بعض المهام الفاشلة في استخدامك المُبلَّغ عنه بينما تظهر أخرى.

* وضع الجلسة: الاستخدام المُبلَّغ عنه هو وقت الساعة الفعلي الذي تكون فيه الجلسة نشطة، بغض النظر عن عدد المهام الفاشلة أو الملغاة.

<span id="view-usage"></span>
## الاستعلام عن الاستخدام الفعلي لحمل العمل
بعد اكتمال حمل العمل، هناك عدة طرق لعرض استخدامه الفعلي:

- شغّل [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) أو [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) في `qiskit-ibm-runtime` الإصدار 0.30 أو أحدث. إذا كنت تستخدم إصدارًا أقدم من `qiskit-ibm-runtime` (>= 0.23 و< 0.30)، لا يزال بإمكانك العثور على الاستخدام في `session.details()["usage_time"]` و`batch.details()["usage_time"]`.
- استخدم [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) لعرض الاستخدام لدُفعة أو جلسة معينة.
- استخدم [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) لعرض الاستخدام لمهمة واحدة.

<span id="instance-usage"></span>
## عرض استخدام المثيل
يمكنك عرض استخدام مثيل على صفحة [المثيلات](https://quantum.cloud.ibm.com/instances)، أو، لمن لديهم الصلاحية المناسبة، صفحة [التحليلات](https://quantum.cloud.ibm.com/analytics). لاحظ أن الصفحتين قد تعرضان أرقام استخدام مختلفة لأنهما تحسبان الاستخدام بطريقة مختلفة.

تعرض صفحة المثيلات الاستخدام في الوقت الفعلي للـ 28 يومًا الأخيرة (متحركة)، حتى الوقت الحالي من اليوم الحالي. أما استخدام صفحة التحليلات فيُعاد حسابه كل ساعة ويشمل آخر 28 يومًا كاملة؛ أي أنه يعرض الاستخدام من الساعة 00:00 قبل 28 يومًا حتى اليوم، عند بداية الساعة.

## تقدير الاستخدام قبل إرسال مهمة
على الرغم من أن الحصول على تقدير محلي دقيق أمر معقد بسبب العمليات الإضافية التي تُجرى لقمع الأخطاء وتخفيفها، يمكنك استخدام هذه الصيغة الأساسية للحصول على تقريب للاستخدام المُقدَّر:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` هو عبء يبلغ تقريبًا 2 ثانية لكل مهمة فرعية. يشمل ذلك عمليات مثل تحميل الحمولة في الإلكترونيات الضابطة. قد تُقسَّم مهمة primitive الخاصة بك إلى عدة مهام فرعية إذا كانت كبيرة جدًا بحيث لا يستطيع محرك التنفيذ معالجتها دفعة واحدة.
- `rep_delay` هو خيار [قابل للتخصيص من قِبَل المستخدم](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay)، والقيمة الافتراضية يحددها `backend.default_rep_delay`، وهي 250 ميكروثانية على معظم backends الكم من IBM. لاحظ أن تقليل `rep_delay` يُقلل من إجمالي وقت تنفيذ QPU، لكن على حساب زيادة معدل خطأ تحضير الحالة؛ راجع دليل [تنفيذ معدل التكرار الديناميكي](/guides/repetition-rate-execution) للمزيد من المعلومات.
- `<circuit length>` هو إجمالي طول التعليمات. تستغرق كل تعليمة وقتًا مختلفًا على QPU، لذا يتفاوت الطول الإجمالي من Circuit إلى أخرى. على سبيل المثال، يمكن أن تستغرق عملية القياس (measurement) 56 ضعف الوقت الذي يستغرقه بوابة `x`. يمكن استخدام `backend.target[<instruction>][<qubit>].duration` لمعرفة المدة الدقيقة لكل تعليمة. الطول النموذجي للـ Circuit يتراوح على الأرجح بين 50-100 ميكروثانية. إذا كنت تستخدم تقنيات قمع الأخطاء أو تخفيفها مع الـ primitives، فقد تُدرج تعليمات إضافية في Circuit الخاصة بك، مما يزيد من إجمالي طول الـ Circuit.
    > **Note:** يُرجع [خيار `scheduler_timing` التجريبي](/guides/visualize-circuit-timing) إجمالي وقت الـ Circuit، لكن هذا ليس الوقت المستخدم للفوترة.
- `<num executions>` هو إجمالي عدد الـ Circuits مضروبًا في عدد اللقطات (shots)، حيث الـ Circuits هي تلك التي تُولَّد بعد بث عناصر PUB.
    - إذا كنت تستخدم تقنيات تخفيف الأخطاء مع الـ primitives، فقد تُشغَّل Circuits إضافية كجزء من عملية التخفيف، مما يزيد من إجمالي عدد عمليات التنفيذ. بالإضافة إلى ذلك، تقنيات تخفيف الأخطاء المتقدمة مثل PEA وPEC تأتي بعبء أعلى بكثير لأنها تتطلب تشغيل Circuits لتعلُّم الضوضاء.
    - يُجمِّع Estimator المراقبات المتبادلة بحسب الكيوبت (Qubit-wise commuting)، مما يُقلل من عدد عمليات التنفيذ.

إذا لم تكن تستخدم أي تقنيات متقدمة لتخفيف الأخطاء أو `rep_delay` مخصصًا، يمكنك استخدام `2+0.00035*<num executions>` كصيغة سريعة.

## الخطوات التالية
> **Tip:** - راجع هذه النصائح: [تقليل وقت تشغيل المهمة](minimize-time).
>     - اضبط [الحد الأقصى لوقت التنفيذ](max-execution-time).
>     - تعلّم كيفية التحويل (Transpile) محليًا في قسم [Transpile](./transpile/).
>     - جرّب دليل [مقارنة إعدادات Transpiler](/guides/circuit-transpilation-settings).